# Pre-Pruning Grid Search

This notebook searches a small set of pre-pruning settings for a decision tree, ranks them by validation F1, and then refits the best configuration.

## Planned Steps

- Load the encoded train and validation sets
- Try all combinations of `max_depth`, `min_samples_split`, and `min_samples_leaf`
- Sort the combinations by validation F1 and print the top 5
- Refit the best tree and report train F1, validation F1, depth, and leaves

In [1]:
from itertools import product
from pathlib import Path

import pandas as pd
from sklearn.metrics import f1_score
from sklearn.tree import DecisionTreeClassifier

TRAIN_PATH = Path('data/processed/encoded_train.csv')
VAL_PATH = Path('data/processed/encoded_validation.csv')

if not TRAIN_PATH.exists():
    TRAIN_PATH = Path('../data/processed/encoded_train.csv')
    VAL_PATH = Path('../data/processed/encoded_validation.csv')

train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)

X_train = train_df.drop(columns=['Approval'])
y_train = train_df['Approval']
X_val = val_df.drop(columns=['Approval'])
y_val = val_df['Approval']

max_depth_values = [3, 5, 7, 10]
min_samples_split_values = [20, 50, 100]
min_samples_leaf_values = [10, 25, 50]

results = []

for max_depth, min_samples_split, min_samples_leaf in product(
    max_depth_values,
    min_samples_split_values,
    min_samples_leaf_values,
):
    model = DecisionTreeClassifier(
        random_state=42,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
    )
    model.fit(X_train, y_train)
    val_pred = model.predict(X_val)
    results.append({
        'max_depth': max_depth,
        'min_samples_split': min_samples_split,
        'min_samples_leaf': min_samples_leaf,
        'val_f1': f1_score(y_val, val_pred, pos_label='Yes'),
    })

results_df = pd.DataFrame(results).sort_values(
    by=['val_f1', 'max_depth', 'min_samples_split', 'min_samples_leaf'],
    ascending=[False, True, True, True],
).reset_index(drop=True)

print('Top 5 combinations by validation F1')
print(results_df.head(5).to_string(index=False))

best = results_df.iloc[0]
best_model = DecisionTreeClassifier(
    random_state=42,
    max_depth=int(best['max_depth']),
    min_samples_split=int(best['min_samples_split']),
    min_samples_leaf=int(best['min_samples_leaf']),
)
best_model.fit(X_train, y_train)

train_pred = best_model.predict(X_train)
val_pred = best_model.predict(X_val)

print('\nBest refit summary')
print(f"train_f1: {f1_score(y_train, train_pred, pos_label='Yes'):.4f}")
print(f"val_f1: {f1_score(y_val, val_pred, pos_label='Yes'):.4f}")
print(f"depth: {best_model.get_depth()}")
print(f"leaves: {best_model.get_n_leaves()}")


Top 5 combinations by validation F1
 max_depth  min_samples_split  min_samples_leaf   val_f1
        10                100                10 0.896797
         7                100                10 0.893372
         7                 50                10 0.892754
         5                 20                50 0.892672
         5                 50                50 0.892672

Best refit summary
train_f1: 0.9041
val_f1: 0.8968
depth: 10
leaves: 57
